# Beyond-ECD Comparison

## Protocol

**Status:** fixed on 2026-10-16, before any model in this notebook was run. Analysis A of `ANALYSIS_PLAN.md` (Amendment 1), recorded in git before execution.

### 1. Framing

The research aim is to test whether multimodal information (slit-lamp findings, Konan morphology, specular microscopy images) improves donor cornea grading beyond ECD alone. This notebook makes that comparison on the primary cohort (non-2023 tissues).

Context that informed this analysis:

- The primary cohort was adopted after the results of notebooks 08–09 were known (see `ANALYSIS_PLAN.md`, timing disclosure).
- A descriptive check in `CLAIMS_AUDIT.md` found that a two-threshold ECD rule scored macro F1 0.800 on these tissues, against 0.758 for the tabular model. This analysis replaces that unmatched check with a matched, donor-grouped comparison against a learned ECD-only model.

### 2. Research question

Do the tabular and hybrid models improve on ECD alone when all are evaluated on the same unseen donors?

### 3. Cohort

| Set | Definition | Expected |
|---|---|---|
| Primary cohort | Labelled, image-matched (two review tissues excluded), year in `Tissue_ID` ≠ 2023; as in notebook 09 | 551 tissues, 312 donors |
| **Evaluation set** | Primary-cohort tissues with a non-missing `ECD_CD_cells_per_mm2` | **470 tissues, 280 donors**; A+ 244, A 159, B 67; years 2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172 |
| No-ECD set | Primary-cohort tissues without ECD (all Konan features missing) | 81 tissues; A+ 33, A 37, B 11 |

The notebook asserts these counts and stops if any differ.

### 4. Models

All models use the Random Forest hyperparameters from notebooks 07–09 (`n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1, max_features="sqrt", class_weight="balanced", random_state=42`). No tuning.

| Model | Inputs | Training tissues in each fold |
|---|---|---|
| **ECD-only** | `ECD_CD_cells_per_mm2` alone, no preprocessing | Training-fold tissues with ECD |
| Tabular | 24 features via `ms.make_preprocessor()` (as notebook 09) | All training-fold tissues of the primary cohort |
| Hybrid | Tabular + frozen ResNet-50 embeddings → StandardScaler → PCA 16 (as notebook 09) | All training-fold tissues of the primary cohort |

- Using the same learner for every model means the comparison varies only the input information.
- The ECD-only model cannot learn from tissues without ECD, so they are excluded from its training.
- Tabular and Hybrid are refitted with notebook 09's exact settings and folds.

**Reproducibility check:** the refitted Tabular and Hybrid out-of-fold predictions must match notebook 09's saved predictions (`non_2023_oof_predictions.csv`) for every tissue and repeat. If they do not, the notebook stops and the discrepancy is recorded in the deviations log before anything else is done.

### 5. Validation

- Existing folds `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to the primary cohort. All fitting, including PCA, uses the training fold only.
- Each model predicts every held-out tissue in the primary cohort that it can. Primary metrics are computed on the evaluation-set tissues within each held-out fold.
- Checks before fitting:
  - no donor in both training and test of any fold;
  - every primary-cohort tissue is tested exactly once per repeat;
  - every held-out fold contains at least one evaluation-set tissue of each grade.

### 6. Comparisons and metrics

| Role | Comparison | Metric |
|---|---|---|
| **Primary P1** | Tabular − ECD-only | Macro F1, per fold, on evaluation-set tissues |
| **Primary P2** | Hybrid − ECD-only | Macro F1, per fold, on evaluation-set tissues |
| Secondary | P1 and P2 | Balanced accuracy, accuracy, grade B recall (unadjusted) |
| Descriptive | Hybrid − Tabular on the evaluation set | Macro F1 (notebook 09 is the test of this comparison; no new test here) |
| Descriptive | All models | Per-grade precision and recall, pooled over repeats |
| Descriptive | ECD threshold rule (Section 8) | Same metrics, no tests |
| Descriptive | Tabular and Hybrid on the no-ECD set | Macro F1 and per-grade recall, pooled over repeats |

### 7. Statistical analysis

- For P1 and P2: mean paired fold-level difference, corrected 95% CI and corrected repeated-CV t-test (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004), 24 degrees of freedom.
  - The variance correction uses n_test / n_train from mean primary-cohort fold sizes.
- **Multiplicity:** Holm correction across P1 and P2 p-values. Confidence intervals are not multiplicity-adjusted and are reported as such.
- **Per-year (descriptive):** for P1 and P2, the pooled difference by year and with each year excluded in turn. 2021 (11 tissues) is flagged as too small to interpret.

### 8. ECD threshold rule (descriptive only)

Rule: B if ECD < 2250, A if 2250 ≤ ECD < 2500, A+ if ECD ≥ 2500.

**Threshold source:**

- These thresholds were **inferred from the 2023 records** after those records had been inspected (2023 A+ minimum ECD 2500; 2023 B maximum 2247).
- No documented eye-bank grading criteria were available when this protocol was written.
- The rule is not fitted to the evaluation tissues but was chosen post hoc, so it is reported **descriptively only**: no tests, and no conclusions rest on it.
- If documented eye-bank criteria are obtained later, they will be evaluated under a separate protocol.

### 9. Interpretation rules

Fixed before running and applied in code.

**Each primary comparison (P1, P2):**

| Result | Reported as |
|---|---|
| Difference > 0 and Holm-adjusted p < 0.05 | Model improved on ECD alone |
| Difference < 0 and Holm-adjusted p < 0.05 | Model performed worse than ECD alone |
| Otherwise | Inconclusive: no evidence of a difference from ECD alone |

**Answer to the research question:**

- *Evidence of improvement beyond ECD alone*: at least one of P1 or P2 is reported as improved.
- *No evidence of improvement beyond ECD alone in this cohort*: neither is improved and neither is worse.
- *Models performed worse than ECD alone*: either is worse. The specific model or models are named.

**Qualifiers:**

- **Grade B recall:** for each comparison, if the mean fold-level grade B recall difference is below −0.02, the result is reported with the reduced B recall and that model is not recommended over ECD alone. As in notebook 09, −0.02 is a project-defined decision rule, not a clinically validated threshold.
- **Per-year:** if a positive pooled difference becomes zero or negative when any single year is excluded, the result is reported as dependent on that year.

**Consequence for analysis B (explainability):** unless an improvement is found, explanations describe model behaviour, not evidence that sources other than ECD add information.

**Other rules:**

- **No changes after results:** no other comparator models, learners, hyperparameters, cohorts, rule thresholds or metrics. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is logged in the deviations log with its reason.

## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| — | None | — |